
# FRED 실업률(UNRATE) 시계열 예측 예제 (비주식)

이 노트북은 **미국 실업률(UNRATE)** 데이터를 FRED에서 직접 불러와
월별 시계열 예측을 수행하는 Colab 예제입니다.

## 학습 목표
- 공개 URL CSV로 시계열 데이터를 온라인에서 바로 수집한다.
- 계절성(12개월)을 고려한 Seasonal Naive와 SARIMA 모델을 비교한다.
- 최근 24개월 테스트 구간에서 MAE/RMSE를 평가한다.
- 향후 12개월 예측값을 생성한다.



## 데이터 설명
- 출처: Federal Reserve Economic Data (FRED)
- 시계열: `UNRATE` (Unemployment Rate)
- 주기: 월별(Monthly)
- 다운로드 URL: `https://fred.stlouisfed.org/graph/fredgraph.csv?id=UNRATE`
- 장점: API 키 없이 즉시 재현 가능, 경제/정책 시계열 수업 예제로 적합


In [ ]:

# 필요 라이브러리 import
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 그래프 기본 크기 설정
plt.rcParams["figure.figsize"] = (12, 4)


In [ ]:

# 1) 온라인에서 데이터 직접 수집 (CSV URL)
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=UNRATE"
raw = pd.read_csv(url)

# 2) 컬럼명 표준화 및 타입 변환
# FRED는 환경/버전에 따라 날짜 컬럼명이 DATE 또는 observation_date일 수 있음
if "DATE" in raw.columns:
    date_col = "DATE"
elif "observation_date" in raw.columns:
    date_col = "observation_date"
else:
    raise KeyError(f"날짜 컬럼을 찾지 못했습니다. 현재 컬럼: {list(raw.columns)}")

value_col = "UNRATE"
if value_col not in raw.columns:
    # 혹시 시리즈 ID 이름이 다르게 내려오면 날짜 컬럼 제외 첫 번째 숫자 컬럼을 사용
    numeric_cols = [c for c in raw.columns if c != date_col]
    if not numeric_cols:
        raise KeyError(f"값 컬럼을 찾지 못했습니다. 현재 컬럼: {list(raw.columns)}")
    value_col = numeric_cols[0]

df = raw.rename(columns={date_col: "date", value_col: "unrate"}).copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["unrate"] = pd.to_numeric(df["unrate"], errors="coerce")

# 3) 결측 제거 후 월 시작 기준(MS)으로 빈도 고정
df = df.dropna(subset=["date", "unrate"]).sort_values("date")
df = df.set_index("date").asfreq("MS")

# 4) 빈도 고정 과정에서 생긴 결측은 선형 보간
# (실무에서는 도메인에 따라 ffill/bfill 등으로 교체 가능)
df["unrate"] = df["unrate"].interpolate(limit_direction="both")

print(f"데이터 기간: {df.index.min().date()} ~ {df.index.max().date()}")
print(f"총 관측치: {len(df):,}개")
df.head()


In [ ]:

# 시계열 전체 추이 확인
ax = df["unrate"].plot(title="US Unemployment Rate (UNRATE)", color="black")
ax.set_ylabel("Unemployment Rate (%)")
ax.set_xlabel("Date")
plt.show()

# 12개월 이동평균(추세 확인용)
roll_12 = df["unrate"].rolling(12).mean()
ax = df["unrate"].plot(label="UNRATE", alpha=0.6)
roll_12.plot(ax=ax, label="12M Rolling Mean", color="tomato")
ax.set_title("UNRATE with 12-Month Rolling Mean")
ax.legend()
plt.show()


In [ ]:

# ACF/PACF 확인 (모델 차수 감 잡기용)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df["unrate"], lags=48, ax=axes[0])
axes[0].set_title("ACF (Original Series)")
plot_pacf(df["unrate"], lags=48, ax=axes[1], method="ywm")
axes[1].set_title("PACF (Original Series)")
plt.tight_layout()
plt.show()


In [ ]:

# 최근 24개월을 테스트로 분리
# 시계열은 반드시 시간 순서를 지켜 분할해야 함 (shuffle 금지)
test_size = 24
train = df.iloc[:-test_size].copy()
test = df.iloc[-test_size:].copy()

print("train:", train.index.min().date(), "~", train.index.max().date(), f"({len(train)}개)")
print("test :", test.index.min().date(), "~", test.index.max().date(), f"({len(test)}개)")


In [ ]:

# 베이스라인 1) Seasonal Naive (12개월 주기 반복)
# 마지막 12개월 패턴을 그대로 다음 12개월(또는 그 이상)에 반복 적용
seasonal_period = 12
last_season = train["unrate"].iloc[-seasonal_period:].values
pred_naive = np.tile(last_season, int(np.ceil(test_size / seasonal_period)))[:test_size]

# 평가 함수

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return mae, rmse

naive_mae, naive_rmse = evaluate(test["unrate"], pred_naive)
print(f"Seasonal Naive - MAE: {naive_mae:.4f}, RMSE: {naive_rmse:.4f}")


In [ ]:

# 모델 2) SARIMA 후보 중 AIC가 가장 낮은 모델 선택
# 너무 큰 그리드는 학습 시간이 길어지므로, 수업용으로 소형 후보 집합 사용
candidates = [
    ((1, 1, 1), (0, 1, 1, 12)),
    ((1, 1, 1), (1, 1, 1, 12)),
    ((2, 1, 1), (1, 1, 0, 12)),
    ((2, 1, 2), (1, 1, 1, 12)),
]

best_cfg = None
best_aic = np.inf
best_result = None

for order, seasonal_order in candidates:
    try:
        model = SARIMAX(
            train["unrate"],
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        result = model.fit(disp=False)
        if result.aic < best_aic:
            best_aic = result.aic
            best_cfg = (order, seasonal_order)
            best_result = result
    except Exception as e:
        print(f"skip {order} x {seasonal_order} -> {e}")

print("Best SARIMA config:", best_cfg)
print("Best AIC:", round(best_aic, 2))


In [ ]:

# 테스트 구간 예측
sarima_pred = best_result.forecast(steps=test_size)
sarima_pred.index = test.index

sarima_mae, sarima_rmse = evaluate(test["unrate"], sarima_pred)
print(f"SARIMA - MAE: {sarima_mae:.4f}, RMSE: {sarima_rmse:.4f}")

# 모델 성능 비교표
result_table = pd.DataFrame(
    {
        "Model": ["Seasonal Naive", "SARIMA"],
        "MAE": [naive_mae, sarima_mae],
        "RMSE": [naive_rmse, sarima_rmse],
    }
).sort_values("RMSE")

result_table


In [ ]:

# 실제값 vs 예측값 시각화
plt.figure(figsize=(12, 5))
plt.plot(train.index, train["unrate"], label="Train", color="gray", alpha=0.7)
plt.plot(test.index, test["unrate"], label="Test (Actual)", color="black")
plt.plot(test.index, pred_naive, label="Seasonal Naive", linestyle="--")
plt.plot(test.index, sarima_pred, label="SARIMA", linestyle="--")
plt.title("UNRATE Forecast: Seasonal Naive vs SARIMA")
plt.xlabel("Date")
plt.ylabel("Unemployment Rate (%)")
plt.legend()
plt.show()


In [ ]:

# 전체 데이터로 최종 재학습 후 향후 12개월 예측
final_model = SARIMAX(
    df["unrate"],
    order=best_cfg[0],
    seasonal_order=best_cfg[1],
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

future_steps = 12
future_forecast = final_model.forecast(steps=future_steps)
future_index = pd.date_range(df.index[-1] + pd.offsets.MonthBegin(1), periods=future_steps, freq="MS")
future_df = pd.DataFrame({"forecast_unrate": future_forecast.values}, index=future_index)

print("향후 12개월 예측")
display(future_df.head())

plt.figure(figsize=(12, 5))
plt.plot(df.index, df["unrate"], label="Historical", color="black")
plt.plot(future_df.index, future_df["forecast_unrate"], label="12M Forecast", linestyle="--", color="tomato")
plt.title("UNRATE: Historical + 12-Month Forecast")
plt.xlabel("Date")
plt.ylabel("Unemployment Rate (%)")
plt.legend()
plt.show()



## 요약
- 본 예제는 주식 데이터가 아닌 **거시경제 시계열(FRED 실업률)**을 사용했다.
- 데이터는 **공개 URL(CSV)에서 온라인으로 직접 수집**했다.
- Seasonal Naive 대비 SARIMA 성능을 MAE/RMSE로 비교해 모델 유효성을 확인했다.
- 동일 구조로 CPI, PCE, 산업생산지수 등 다른 FRED 시계열에도 확장 가능하다.
